# Pancancer enrichment analysis step 5: Find enriched pathways using GSEApy with Reactome data

## Setup

In [1]:
import cptac
import cptac.utils as ut
import pandas as pd
import numpy as np
import gseapy as gp
import os
import datetime
import glob

In [2]:
TIME_START = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
NUM_PERMUTATIONS = 10000

STEP04_DIR = "step04_outputs"
STEP05_DIR = "step05_outputs"

# Create log dir and file
LOG_DIR = "step05_checkpoints"
LOG_FILE = f"{TIME_START}_{NUM_PERMUTATIONS}_perms_wst0.log"
LOG_FILE_PATH = os.path.join(LOG_DIR, LOG_FILE)

if not os.path.isdir(LOG_DIR):
    os.mkdir(LOG_DIR)
    
with open(LOG_FILE_PATH, 'w') as fp: 
    fp.write(f"{TIME_START}\n")
    fp.write(f"{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - Started step 5 with {NUM_PERMUTATIONS} permutations.\n")

GSEAPY_DIR_PATH = os.path.join(STEP05_DIR, "gseapy")

if not os.path.isdir(STEP05_DIR):
    os.mkdir(STEP05_DIR)

## Prepare data

In [3]:
# Read in the files from step 4
step04_files = glob.glob(f"{STEP04_DIR}{os.sep}*")
prot_dict = {}
for file in step04_files:
    name = file.split(os.sep)[1].split("_")[0]
    prot_dict[name] = pd.read_csv(file, sep='\t')

In [4]:
prot_dict.keys()

dict_keys(['hnscc', 'lscc', 'colon', 'endometrial', 'luad', 'ccrcc', 'gbm', 'ovarian'])

## Run enrichment analysis

In [5]:
def gseapy_enrich_reactome(method, proteomics_dictionary):
    
    with open(LOG_FILE_PATH, 'a') as fp: 
        fp.write(f"\n{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - Started {method} method.\n")
    
    output_file = f"enrichment_gseapy_reactome_{TIME_START}_{NUM_PERMUTATIONS}_perms_wst0_{method}.tsv"
    output_path = os.path.join(STEP05_DIR, output_file)
    
    # For each cancer, find enriched pathways.
    all_enrichments = pd.DataFrame()

    for cancer_type in proteomics_dictionary.keys():

        prot = proteomics_dictionary[cancer_type]
        samples = prot.columns[~prot.columns.isin(["NAME"])]
        cls_list = np.where(samples.str.endswith(".N"), "Normal", "Tumor").tolist()

        gs_res = gp.gsea(
            data=prot,
            gene_sets="gene_set_libraries/Reactome_2020.gmt",
            cls=cls_list,
            permutation_type="phenotype",
            permutation_num=NUM_PERMUTATIONS,
            min_size=1,
            max_size=500, 
            outdir=GSEAPY_DIR_PATH,
            no_plot=True,
            method=method,
            processes=1,
            seed=0,
            weighted_score_type=0,
            ascending=False)

        cancer_enriched = gs_res.res2d.assign(cancer_type=cancer_type)
        all_enrichments = all_enrichments.append(cancer_enriched)
        all_enrichments.to_csv(output_path, sep="\t")

        # Log that we finished this cancer type
        with open(LOG_FILE_PATH, 'a') as fp: 
            fp.write(f"{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - Finished {cancer_type} data.\n")
            
    with open(LOG_FILE_PATH, 'a') as fp: 
        fp.write(f"\n{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - Finished {method} method.\n")

In [6]:
gseapy_enrich_reactome("abs_signal_to_noise", prot_dict)
gseapy_enrich_reactome("signal_to_noise", prot_dict)
gseapy_enrich_reactome("t_test", prot_dict)
gseapy_enrich_reactome("ratio_of_classes", prot_dict)
gseapy_enrich_reactome("diff_of_classes", prot_dict)
gseapy_enrich_reactome("log2_ratio_of_classes", prot_dict)

2020-06-18 10:44:59,487 Warning: Input data contains NA, filled NA with 0
2020-06-18 11:18:55,962 Warning: Input data contains NA, filled NA with 0
2020-06-18 11:47:43,039 Warning: Input data contains NA, filled NA with 0
2020-06-18 12:02:19,916 Warning: Input data contains NA, filled NA with 0
2020-06-18 12:29:18,962 Warning: Input data contains NA, filled NA with 0
2020-06-18 12:54:57,081 Warning: Input data contains NA, filled NA with 0
2020-06-18 13:26:17,053 Warning: Input data contains NA, filled NA with 0
2020-06-18 13:45:17,142 Warning: Input data contains NA, filled NA with 0
2020-06-18 14:08:46,970 Warning: Input data contains NA, filled NA with 0
2020-06-18 14:38:35,646 Warning: Input data contains NA, filled NA with 0
2020-06-18 14:55:48,175 Warning: Input data contains NA, filled NA with 0
2020-06-18 15:07:52,518 Warning: Input data contains NA, filled NA with 0
2020-06-18 15:24:30,323 Warning: Input data contains NA, filled NA with 0
2020-06-18 15:40:43,606 Warning: Input